In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup, AutoConfig
from tqdm import tqdm

import optuna

import warnings
warnings.filterwarnings('ignore')

## EDA

In [2]:
ner_data_path = Path('../data/ner/animals_ner_dataset.csv')
df = pd.read_csv(ner_data_path)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2582 entries, 0 to 2581
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sentence  2582 non-null   str  
 1   animal    2582 non-null   str  
dtypes: str(2)
memory usage: 40.5 KB


In [3]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)
df.info()

<class 'pandas.DataFrame'>
Index: 2503 entries, 0 to 2526
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sentence  2503 non-null   str  
 1   animal    2503 non-null   str  
dtypes: str(2)
memory usage: 58.7 KB


In [4]:
df.head()

,sentence,animal
0,This scene includes a butterfly resting on a y...,butterfly
1,The shot focuses on a butterfly resting on a y...,butterfly
2,You can see a butterfly resting on a yellow fl...,butterfly
3,The camera catches a butterfly resting on a ye...,butterfly
4,The subject of the photo is a butterfly restin...,butterfly


In [5]:
df['animal'].value_counts()

animal
butterfly    251
cat          251
cow          251
dog          251
horse        251
sheep        250
spider       250
chicken      249
elephant     249
squirrel     249
animal         1
Name: count, dtype: int64

In [6]:
df[df['animal'] == 'animal']

,sentence,animal
1940,sentence,animal


In [7]:
df = df[df['animal'] != 'animal']
df['animal'].value_counts()

animal
butterfly    251
cat          251
cow          251
dog          251
horse        251
sheep        250
spider       250
chicken      249
elephant     249
squirrel     249
Name: count, dtype: int64

In [ ]:
# Split the dataset into train, validation, and test sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['animal'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['animal'])
print(f"Train set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

Train set size: 2001
Validation set size: 250
Test set size: 251


## NER label preparation

In [14]:
# Function to create NER examples
def make_ner_example(sentence, animal):
    tokens = re.findall(r'\w+|[^\w\s]', sentence)
    labels = ['O'] * len(tokens)

    animal_lower = animal.lower()
    for idx, token in enumerate(tokens):
        if token.lower() == animal_lower:
            labels[idx] = 'B-ANIMAL'

    return tokens, labels


In [15]:
tokens, labels = make_ner_example("The cat is on the roof.", "cat")
print("Tokens:", tokens)
print("Labels:", labels)

Tokens: ['The', 'cat', 'is', 'on', 'the', 'roof', '.']
Labels: ['O', 'B-ANIMAL', 'O', 'O', 'O', 'O', 'O']


In [19]:
train_df[['tokens', 'labels']] = train_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)
val_df[['tokens', 'labels']] = val_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)
test_df[['tokens', 'labels']] = test_df.apply(
    lambda row: make_ner_example(row['sentence'], 
    row['animal']), axis=1, result_type='expand'
)

print(train_df.head())

                                               sentence   animal  \
1245  In the photo, a horse is standing beside a sto...    horse   
390   The subject of the photo is a chicken standing...  chicken   
458   The image captures the chicken foraging near a...  chicken   
511   The picture features a chicken resting beside ...  chicken   
1551  You can see a spider waiting near a wall crack...   spider   

                                                 tokens  \
1245  [In, the, photo, ,, a, horse, is, standing, be...   
390   [The, subject, of, the, photo, is, a, chicken,...   
458   [The, image, captures, the, chicken, foraging,...   
511   [The, picture, features, a, chicken, resting, ...   
1551  [You, can, see, a, spider, waiting, near, a, w...   

                                                 labels  
1245  [O, O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O,...  
390   [O, O, O, O, O, O, O, B-ANIMAL, O, O, O, O, O,...  
458   [O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O, O,...  
511 

In [21]:
# Create label mappings
label_to_id = {'O': 0, 'B-ANIMAL': 1}
id_to_label = {0: 'O', 1: 'B-ANIMAL'}

In [24]:
# 
train_df['label_ids'] = train_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])
val_df['label_ids'] = val_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])
test_df['label_ids'] = test_df['labels'].apply(lambda labels: [label_to_id[label] for label in labels])

In [25]:
train_df.head()

,sentence,animal,tokens,labels,label_ids
1245,"In the photo, a horse is standing beside a sto...",horse,"[In, the, photo, ,, a, horse, is, standing, be...","[O, O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O,...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
390,The subject of the photo is a chicken standing...,chicken,"[The, subject, of, the, photo, is, a, chicken,...","[O, O, O, O, O, O, O, B-ANIMAL, O, O, O, O, O,...","[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, ..."
458,The image captures the chicken foraging near a...,chicken,"[The, image, captures, the, chicken, foraging,...","[O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O, O,...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]"
511,The picture features a chicken resting beside ...,chicken,"[The, picture, features, a, chicken, resting, ...","[O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O, O,...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1551,You can see a spider waiting near a wall crack...,spider,"[You, can, see, a, spider, waiting, near, a, w...","[O, O, O, O, B-ANIMAL, O, O, O, O, O, O, O, O,...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


## Model initialization

In [28]:
model_name = 'bert-base-cased'
num_classes = len(label_to_id)

config = AutoConfig.from_pretrained(model_name, num_labels=num_classes)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=num_classes)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 19668.13it/s]
BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe